# [Baseline] RandomForest — 학습

KBO 투구 하나가 **제구 성공** 투구일 확률을 예측하는 베이스라인입니다.

- **입력**: `test.csv` 의 47개 컬럼 (경기 상황, 투수·타자의 직전까지 누적 기록 등)
- **출력**: 제구 성공 확률 (0 이상 1 이하의 실수)
- **평가지표**: Brier Skill Score

`trackman_history.csv` 는 이 베이스라인에서 사용하지 않습니다. 2019~2024 과거 로그
179만 행이 그대로 남아 있으니 직접 활용해 보세요.

이 노트북은 모델을 **학습**하여 `./model/rf.pkl` 로 저장합니다. 저장한 모델은
추론용 `script.py` 와 함께 `baseline_submit.zip` 으로 묶어 제출합니다.

## 1. 라이브러리 불러오기

데이터 처리(pandas)와 모델 학습(scikit-learn)에 필요한 라이브러리를 불러옵니다.
`joblib` 은 학습한 모델을 파일로 저장할 때 사용합니다.

In [ ]:
import os
import time

import joblib
import numpy as np  # [신규] 트렌드 보정(logit shift)용
import pandas as pd
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin  # [변경] ClassifierMixin 추가 (TrendCalibrator용)
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier  # [신규] HGB 비교 추가
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder  # [변경] OrdinalEncoder -> OneHotEncoder

DATA_DIR = "./data"

ID = "row_id"
TARGET = "control_success"
# [변경, 2026-08-20] pitcher_team_id/batter_team_id 추가 — 13개 범주(값 12~25)인데
# 그동안 NUM_COLS로 새어 들어가 순서형 숫자로 취급되고 있었음(median impute만 됨,
# 원-핫 안 됨). base_state/game_type/top_bottom에서 이미 고쳤던 것과 같은 종류의
# 결함. 재현 스크립트로 확인: HGB+OneHot 기준 mean 908.48 -> 934.02,
# fold=2022(+47.7)/fold=2024(+28.9) 동시 개선 — IMPROVEMENT_NOTES.md
# "team_id 범주형 처리 누락 수정" 참고.
CAT_COLS = ["top_bottom", "game_type", "base_state", "pitcher_team_id", "batter_team_id"]

## 2. 데이터 불러오기

`train.csv` 는 2019~2024 시즌이고 평가 데이터는 2025 시즌입니다.

사용할 피처 목록은 `test.csv` 가 정합니다. `train.csv` 에만 있는 컬럼을 학습에 넣으면
평가 시점에 그 컬럼이 없어 추론이 실패하기 때문입니다.

In [23]:
test_cols = pd.read_csv(os.path.join(DATA_DIR, "test.csv"),
                        encoding="utf-8-sig", nrows=0).columns
FEATURES = [c for c in test_cols if c != ID]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"),
                    encoding="utf-8-sig", usecols=FEATURES + [TARGET])

print("train:", train.shape, "| 피처:", len(FEATURES),
      f"(범주형 {len(CAT_COLS)}, 수치형 {len(NUM_COLS)})")
print("시즌:", train["season"].min(), "~", train["season"].max())
print(f"제구 성공률: {train[TARGET].mean():.4f}")

train: (1475092, 48) | 피처: 47 (범주형 3, 수치형 44)
시즌: 2019 ~ 2024
제구 성공률: 0.5238


## 3. 전처리 정의 — asof_* 스무딩 + 파생 피처 + 인코딩

범주형 3개(`top_bottom`, `game_type`, `base_state`)는 원-핫 인코딩하고, 수치형 컬럼의
결측값은 중앙값으로 채웁니다 (안전망 — 스무딩 이후엔 결측이 거의 남지 않습니다).

**`asof_*` 표본수 기반 Bayesian smoothing.** `asof_pitcher_n`, `asof_batter_n`,
`asof_pitcher_pitchmix_n`이 0인 행(신인/이적 초반, 학습 데이터 기준 각각 792/830/792행)은
관련 rate 컬럼이 결측입니다. 이걸 전체 median으로 채우면 신인에게 베테랑 수준 값을
부여하는 왜곡이 생기므로, 표본수를 신뢰도로 삼아 `(n*rate + k*prior) / (n+k)` 형태로
스무딩합니다 (`k=50`). cold-start 행은 `control_success` 평균이 오히려 전체보다 높았던
패턴을 모델이 학습하도록 `is_pitcher_cold_start` / `is_batter_cold_start` 플래그도
추가합니다. `asof_pitcher_prev1/3/5_game_*` 결측은 같은 지표의 스무딩된 누적값으로
대체합니다.

**[신규] 투수-타자 능력 차이 / 상황 압박 파생 피처.** RF는 얕은 트리(depth=10)라
"투수 능력 - 타자 능력" 같은 상호작용을 스스로 찾으려면 여러 번 split을 거쳐야 합니다.
`pitcher_batter_success_diff`, `pitcher_batter_middle_diff`(스무딩된 rate 차이),
카운트 압박(`count_pressure`, `is_full_count`, `is_two_strike_pressure`), 득점권 여부
(`is_scoring_position`), 좌우 매치업(`same_hand_matchup`)을 명시적으로 만들어 넣습니다.

**[변경] 범주형 인코딩: OrdinalEncoder → OneHotEncoder.** `base_state`, `game_type`,
`top_bottom`은 명목형(순서 없는 범주)인데 기존 OrdinalEncoder는 임의의 정수 순서를
부여했습니다. 카디널리티가 작아서(8/2/2, 총 12개 컬럼) OneHot으로 바꿔도 비용이 크지
않고, 트리가 특정 범주를 한 번의 split으로 분리할 수 있어 더 효율적일 수 있습니다.

스무딩 → 파생 피처 → 인코딩/대치 순서로 전부 파이프라인 안에 넣어서, 추론할 때도 학습
때와 동일한 로직이 그대로 따라가게 합니다.

In [ ]:
# asof_* rate 컬럼의 cold-start 결측치를 표본수 기반으로 스무딩
class AsofRateSmoother(BaseEstimator, TransformerMixin):
    """표본수(n)가 작을수록 prior(학습 데이터 평균) 쪽으로, 클수록 실제 관측값
    쪽으로 끌어당기는 (n*rate + k*prior) / (n+k) 형태의 empirical Bayes smoothing.
    n=0(결측)이면 그대로 prior 값이 된다.
    """

    RATE_GROUPS = [
        ("asof_pitcher_n", ["asof_pitcher_success_rate", "asof_pitcher_reverse_rate",
                             "asof_pitcher_middle_rate", "asof_pitcher_ball_rate",
                             "asof_pitcher_strike_rate"]),
        ("asof_batter_n", ["asof_batter_success_rate", "asof_batter_middle_rate"]),
        ("asof_pitcher_pitchmix_n", ["asof_pitcher_fastball_rate",
                                      "asof_pitcher_breaking_rate",
                                      "asof_pitcher_offspeed_rate"]),
    ]
    # 직전 N경기 지표가 없으면(첫 등판 이후 두 번째 경기 전까지) 누적 지표로 대체
    PREV_GAME_FALLBACK = [
        ("asof_pitcher_prev1_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev3_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev5_game_success_rate", "asof_pitcher_success_rate"),
        ("asof_pitcher_prev1_game_middle_rate", "asof_pitcher_middle_rate"),
        ("asof_pitcher_prev3_game_middle_rate", "asof_pitcher_middle_rate"),
        ("asof_pitcher_prev5_game_middle_rate", "asof_pitcher_middle_rate"),
    ]
    SMOOTHING_K = 50  # prior를 표본 몇 개어치로 신뢰할지

    def fit(self, X, y=None):
        # prior는 fold의 학습 데이터에서만 계산 (검증/평가 데이터 누수 방지)
        self.priors_ = {
            c: X[c].mean()
            for _, rate_cols in self.RATE_GROUPS for c in rate_cols
        }
        return self

    def transform(self, X):
        X = X.copy()
        X["is_pitcher_cold_start"] = (X["asof_pitcher_n"] == 0).astype(int)
        X["is_batter_cold_start"] = (X["asof_batter_n"] == 0).astype(int)

        for n_col, rate_cols in self.RATE_GROUPS:
            n = X[n_col]
            for c in rate_cols:
                raw = X[c].fillna(0)
                X[c] = (n * raw + self.SMOOTHING_K * self.priors_[c]) / (n + self.SMOOTHING_K)

        for prev_col, fallback_col in self.PREV_GAME_FALLBACK:
            X[prev_col] = X[prev_col].fillna(X[fallback_col])

        return X


# [신규] asof_smooth 다음 단계 — 기존 컬럼을 조합한 파생 피처 추가
class DerivedFeatureBuilder(BaseEstimator, TransformerMixin):
    """투수-타자 능력 차이, 카운트/주자 압박, 좌우 매치업 등 도메인 지식 기반
    상호작용을 명시적으로 만든다. row별 계산이라 fit에서 저장할 상태가 없다.
    AsofRateSmoother 다음에 실행되어야 한다 (스무딩된 rate 값을 사용하므로).
    """

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X["pitcher_batter_success_diff"] = (
            X["asof_pitcher_success_rate"] - X["asof_batter_success_rate"])
        X["pitcher_batter_middle_diff"] = (
            X["asof_pitcher_middle_rate"] - X["asof_batter_middle_rate"])

        X["count_pressure"] = X["balls_before"] - X["strikes_before"]
        X["is_full_count"] = ((X["balls_before"] == 3) & (X["strikes_before"] == 2)).astype(int)
        X["is_two_strike_pressure"] = (X["strikes_before"] == 2).astype(int)
        X["is_scoring_position"] = ((X["runner_on_2b"] == 1) | (X["runner_on_3b"] == 1)).astype(int)

        X["same_hand_matchup"] = (X["pitcher_hand"] == X["batter_hand"]).astype(int)
        return X


DERIVED_NUM_COLS = [
    "pitcher_batter_success_diff", "pitcher_batter_middle_diff",
    "count_pressure", "is_full_count", "is_two_strike_pressure",
    "is_scoring_position", "same_hand_matchup",
]

# [변경] 스무딩 플래그 2개 + 파생 피처 7개를 수치형 컬럼에 추가
NUM_COLS_EXT = NUM_COLS + ["is_pitcher_cold_start", "is_batter_cold_start"] + DERIVED_NUM_COLS

# [변경] OrdinalEncoder -> OneHotEncoder (명목형 범주라 순서 부여가 부적절했음)
# [변경, 2026-08-20] CAT_COLS에 team_id 2개 추가되어 카디널리티 8/2/2/13/13,
# 총 38개 원-핫 컬럼 — 여전히 비용 부담 없음.
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_COLS),
    ("num", SimpleImputer(strategy="median"), NUM_COLS_EXT),
])

## 3-1. 시즌 트렌드 외삽 보정 (Trend Calibration)

**배경**: fold=2023 진단(아래 4번 섹션 참고)에서 확인했듯, 모델이 `season`을 정수
피처로만 다뤄서 훈련 구간 밖의 하락 추세를 못 뻗어나가고(extrapolate), 그 결과
예측 평균이 실제보다 항상 높게 나옵니다. Recency weighting은 "가장 최근 관측 시즌
쪽으로 앵커링"만 할 뿐 그 이후의 추가 하락은 반영하지 못해 효과가 미미했습니다
(`IMPROVEMENT_NOTES.md` "recency weighting 시도" 참고). 반면 시즌별 성공률에 단순
선형회귀를 적용하면 다음 시즌 값을 꽤 정확히 외삽할 수 있음을 이미 확인했습니다
(2019~2023만으로 2024를 외삽하면 0.4918, 실제 0.4861 — 오차 0.0057).

**방식**: `TrendCalibrator`가 `make_model()`이 만드는 파이프라인 전체를 감싸는
메타 추정기로 동작합니다.
1. `fit()`에서 내부 파이프라인을 평소처럼 학습한 뒤, (a) **학습 데이터**의 시즌별
   실제 성공률에 1차 선형회귀를 적합해 추세선(`slope_`, `intercept_`)을 구하고,
   (b) 내부 파이프라인 자신의 학습 데이터 평균 예측 확률(`train_pred_mean_`)을 보정
   기준점으로 저장합니다. walk-forward CV의 각 fold가 자기 학습 데이터로만 추세선/기준점을
   적합하므로 검증 데이터 누수는 없습니다.
2. `predict_proba()`에서는 각 행의 `season`을 추세선에 대입한
   `target_rate`와 `train_pred_mean_`의 차이를 logit 공간에서 구해 모든 예측에
   동일하게 shift(순위 보존, 수준만 이동).

**[버그 수정, 2026-08-18] 왜 "전체 기간 평균"이 아니라 "마지막 학습 시즌"인가**:
처음엔 기준점을 학습 전체 기간(예: 2019~2023)의 평균 예측 확률로 잡았는데,
fold=2024에서 CV 점수가 597.54 → 361로 급락하는 회귀가 발생했습니다. 원인은 RF/HGB
같은 트리 모델이 학습 범위 밖의 `season` 값을 진짜로 extrapolate하지 못하고 마지막
분기점과 같은 leaf로 취급한다는 점입니다 — 그래서 `season=2024`(학습 범위 밖) 행의
raw 예측은 이미 "마지막 학습 시즌(2023) 수준"으로 저절로 수렴해 있었습니다(실측:
raw 예측 평균 0.4980 ≈ 2023 실제 평균 0.4999). 기준점을 전체 기간 평균(0.5313, 더
높음)으로 잡으면 이미 낮아져 있는 raw 예측을 한 번 더 크게 끌어내려 과잉보정하게
됩니다. 기준점을 "마지막 학습 시즌 자체의 raw 예측 평균"으로 바꾸면 필요한 보정
폭이 정확하게 작아집니다 (재현 스크립트로 검증: fold=2024 597.54 → 650.35로 개선).

`season`은 이미 `test.csv`에 있는 컬럼이라(2025 고정값) 새 피처가 아니고, 다른
행의 통계를 쓰지 않으므로 대회 규칙에 저촉되지 않습니다. **`build_features()`는
수정 불필요**하지만, `AsofRateSmoother`/`DerivedFeatureBuilder`와 마찬가지로
`TrendCalibrator`도 joblib pickle이 클래스 정의를 요구하므로 `script.py`에 동일하게
복사해 넣어야 합니다.

**[2026-08-19] 반기 단위 시간축 실험 — 되돌림**: `season + 0.5*(game_month>=7)`로
시간축을 세분화하는 실험을 CV에서 좋은 결과(mean 908.5→952.9)를 보고 제출했으나
실제 점수가 862.9 → 851로 하락(-11.9). `game_type`(R=1군/F=2군 퓨처스) 기준으로
따로 추세선을 적합하는 실험도 CV에서부터 더 나빠짐(mean 827.4, F 표본이 적어
추세선이 노이즈에 휘둘림). 두 실험 다 `IMPROVEMENT_NOTES.md`에 기록하고 이 섹션은
검증된 season 단위 버전(862.9점)으로 되돌림 — 자세한 내용은 노트 참고.

다음 섹션(walk-forward CV)에서 `USE_TREND_CALIBRATION` 플래그로 적용 전/후를 바로
비교합니다.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


class TrendCalibrator(BaseEstimator, ClassifierMixin):
    """시즌별 성공률의 선형 하락 추세를 학습 구간 밖(다음 시즌)으로 외삽해서,
    감싼 파이프라인의 예측 확률 평균 수준을 그 추세선 쪽으로 이동시키는 사후 보정.

    내부 파이프라인(asof_smooth/derived/pre/clf)은 그대로 두고, season별 실제
    성공률 선형회귀로 구한 추세와 파이프라인 자신의 raw 예측 평균의 차이를 logit
    shift로 모든 예측에 동일하게 적용한다 — 행별 상대 순위는 보존되고 전체 수준만
    옮겨간다.

    [버그 수정, 2026-08-18] 보정 기준점을 "학습 전체 기간 평균 예측"이 아니라
    "마지막 학습 시즌 자체에 대한 raw 예측 평균"으로 잡는다. RF/HGB 같은 트리
    모델은 학습 범위 밖 season 값을 extrapolate하지 못하고 마지막 분기점과 같은
    leaf로 취급하므로, 검증/평가 대상 시즌(학습 범위 밖)의 raw 예측은 이미
    "마지막 학습 시즌 수준"으로 저절로 수렴해 있다. 기준점을 전체 기간 평균으로
    잡으면 이미 낮아져 있는 raw 예측을 한 번 더 크게 끌어내려 과잉보정(overshoot)
    하게 된다 — 실제로 fold=2024에서 597.54 -> 361로 점수가 급락한 원인이었다.

    [2026-08-19] 시간축을 반기 단위(`season+0.5*half`)로 세분화하는 실험, game_type
    (R/F)별로 추세선을 따로 적합하는 실험을 모두 시도했으나 둘 다 CV/실전에서
    이 season 단일 버전(실제 제출 862.9점)보다 못해 되돌림 — `IMPROVEMENT_NOTES.md`
    참고.
    """

    def __init__(self, pipeline, season_col="season"):
        self.pipeline = pipeline
        self.season_col = season_col

    def fit(self, X, y, **fit_params):
        self.pipeline.fit(X, y, **fit_params)

        seasons = X[self.season_col]
        last_season = seasons.max()
        raw_train_pred = self.pipeline.predict_proba(X)[:, 1]
        # [변경] 전체 기간 평균이 아니라 마지막 학습 시즌 행들만의 raw 예측 평균
        self.anchor_pred_mean_ = raw_train_pred[(seasons == last_season).values].mean()

        # 시즌별 실제 성공률에 1차 선형회귀 -> 추세선
        # (fold의 학습 데이터로만 적합하므로 검증/평가 데이터 누수 없음)
        season_means = y.groupby(seasons).mean()
        if season_means.shape[0] < 2:
            # 학습 시즌이 1개뿐이면 추세선을 그릴 수 없어 평균값으로 대체(사실상 보정 없음)
            self.slope_, self.intercept_ = 0.0, float(season_means.mean())
        else:
            self.slope_, self.intercept_ = np.polyfit(
                season_means.index.values, season_means.values, 1)
        return self

    def _target_rate(self, X):
        return self.slope_ * X[self.season_col].values + self.intercept_

    def predict_proba(self, X):
        raw = self.pipeline.predict_proba(X)[:, 1]
        delta = logit(self._target_rate(X)) - logit(self.anchor_pred_mean_)  # [변경]
        calibrated = sigmoid(logit(raw) + delta)
        return np.column_stack([1 - calibrated, calibrated])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

## 4. 모델 학습과 검증 — Walk-forward CV + Recency Weighting

트리 깊이 10, 잎 노드 최소 샘플 200으로 얕게 제한합니다 (baseline 하이퍼파라미터, 이후 튜닝 대상).

기존에는 2019~2023으로 학습해 2024 한 시즌만 검증했습니다. `train.csv`의 연도별
제구 성공률이 2019년 0.565에서 2024년 0.486까지 꾸준히 낮아지는 추세라, 단일 연도
검증은 운(luck)에 따라 점수가 흔들릴 수 있습니다. 대신 시즌 기준
walk-forward(rolling-origin) 방식으로 과거 시즌들로 학습해 그다음 시즌을 맞히는 상황을
3번 반복합니다.

- Fold 1: train 2019~2021 → val 2022
- Fold 2: train 2019~2022 → val 2023
- Fold 3: train 2019~2023 → val 2024 (기존 baseline과 동일한 split)

**[신규] Recency weighting.** fold별 train/val Brier을 직접 비교해보니, val 점수가
fold마다 크게 흔들리는 원인이 overfitting이 아니라 **트렌드 추정 실패**였습니다:

| val 시즌 | 예측 평균 | 실제 평균 | 차이 | val 점수 |
|---|---|---|---|---|
| 2022 | 0.5305 | 0.5289 | +0.0016 | 2176.6 |
| 2023 | 0.5216 | 0.5000 | +0.0216 | 0.0 |
| 2024 | 0.5003 | 0.4861 | +0.0142 | 476.7 |

TRAIN 점수는 세 fold 모두 2173~2396으로 안정적인데, VAL만 요동칩니다. RF가 `season`을
정수 피처로만 다뤄서 훈련 구간 밖의 하락 추세를 못 뻗어나가고(extrapolate), 예측
평균이 실제보다 계속 높게 나오는 게 원인입니다(과대예측 폭이 클수록 점수가 더 크게
깎임). 이를 완화하기 위해 `RandomForestClassifier.fit()`에 `sample_weight`를 줘서
오래된 시즌의 영향력을 줄이고 최근 시즌 비중을 높입니다 (`season_sample_weight()`,
`half_life=2` — 2년마다 가중치 절반). 학습 데이터의 평균이 예측 대상 시즌에 더 가깝게
이동하길 기대합니다.

fold별 Brier Skill Score와 평균/표준편차를 함께 확인합니다.

**[신규] 모델 비교 — RandomForest vs HistGradientBoosting.** RF는 얕은 트리를 단순 평균 내는 구조라 Brier(확률 보정)에는 다소 불리할 수 있습니다. 반면 `HistGradientBoostingClassifier`는 log-loss를 직접 최적화하며 순차적으로 잔차를 학습하기 때문에 확률 보정이 더 좋고, 얕은 트리만으로도 상호작용을 잘 잡아내는 경향이 있습니다. scikit-learn 내장이라 `requirements.txt`에 새 의존성을 추가하지 않아도 됩니다. `make_model(model_type=...)`으로 `"rf"`/`"hgb"`를 골라서 동일한 전처리·walk-forward CV·recency weighting 위에서 그대로 비교할 수 있습니다.

**[2026-08-19, 시도 후 되돌림] CatBoost + OneHot 비교.** RF/HGB/LightGBM/XGBoost/CatBoost
5개를 season TrendCalibrator 위에서 비교했을 때 CatBoost가 CV에서 가장 좋았음
(mean 908.5→956.1, fold=2024 650.4→703.3). 하지만 **실제 제출 862.2점으로 HGB의
862.9점과 사실상 동률(-0.7)** — 외부 의존성만 늘고 실이득이 없어 코드는 HGB로 되돌림.

**[2026-08-20, 시도 후 되돌림] CatBoost 네이티브 범주형 처리.** OneHot 대신
CatBoost의 `cat_features`(ordered target statistics)를 써봤더니 CV는 역대 최고
(mean 908.48→970.55, fold=2024 650.35→**732.12**)였지만, **실제 제출 855점으로
오히려 862.9 대비 -7.9 하락** — CV 개선폭이 가장 컸는데 실전 하락폭도 가장 컸던
사례. HGB 그리드서치 실패 사례와 같은 계열의 "과거 6개 시즌 잡음 패턴에 더 밀착
(암기)" 가설에 무게가 실림. 코드는 다시 HGB로 되돌림.

**[2026-08-20, 채택] team_id 범주형 처리 누락 수정.** 두 번의 "모델/인코딩을 더
정교하게" 시도가 연달아 실패한 뒤, 방향을 바꿔서 "실제 결함을 고치는" 쪽을 찾다가
발견 — `CAT_COLS`에 `pitcher_team_id`/`batter_team_id`가 빠져 있어서 13개 범주인
팀 ID가 그동안 순서형 숫자로 새고 있었습니다(`base_state`/`game_type`/`top_bottom`
때 고쳤던 것과 같은 종류의 문제). `CAT_COLS`에 추가해서 원-핫으로 고친 결과
CV mean 908.48→934.02, **fold=2022(+47.7)와 fold=2024(+28.9)가 동시에 개선**
— 지금까지 대부분 한쪽 fold만 좋아지던 것과 다른, 더 일관된 신호. 새 모델 종류를
들이는 게 아니라 기존 파이프라인의 누락을 고치는 성격이라 채택. 비교 표와 전체
맥락은 `IMPROVEMENT_NOTES.md` "team_id 범주형 처리 누락 수정" 참고.

In [ ]:
# ── [변경] 단일 split(2019~2023 학습 / 2024 검증) → walk-forward CV로 교체 ──

# [신규] model_type으로 RF <-> HistGradientBoosting을 바로 교체해서 비교
# [2026-08-19] CatBoost를 OneHot 인코딩 위에서 비교(CV mean 908.5→956.1, 실제 제출
# 862.2점으로 862.9점과 동률) → 되돌림.
# [2026-08-20] CatBoost 네이티브 범주형 처리(OneHot 대신 cat_features)도 시도 —
# CV는 역대 최고(mean 970.55, fold=2024 732.12)였지만 **실제 제출 855점으로 오히려
# 862.9 대비 -7.9 하락**. CV 개선폭이 가장 컸는데 실전 하락폭도 가장 컸던 사례.
# 코드(catboost import, preprocessor_native_cat, catboost_native 분기)는 전부
# 제거하고 HGB로 되돌림 — 대신 이 기회에 발견한 실제 결함(CAT_COLS에
# pitcher_team_id/batter_team_id가 빠져 순서형 숫자로 새던 문제)을 고쳐서 얹음.
# CV: mean 908.48 -> 934.02, fold=2022(+47.7)/fold=2024(+28.9) 동시 개선.
# 자세한 내용은 IMPROVEMENT_NOTES.md "네이티브 범주형 처리 실험",
# "team_id 범주형 처리 누락 수정" 참고.
MODEL_TYPE = "hgb"  # "rf"(기존 baseline) 또는 "hgb"(채택, 810→862.9→team_id 수정 후 재검증 중)

# [신규] 시즌 트렌드 외삽 보정(TrendCalibrator) on/off — CV에서 적용 전/후를 바로 비교
USE_TREND_CALIBRATION = True


def make_model(model_type=MODEL_TYPE, calibrate=USE_TREND_CALIBRATION):
    """fold마다 새로 학습할 모델(학습 전 상태)을 만든다.

    전처리(asof_smooth/derived/pre)는 두 모델이 동일하게 공유하므로
    model_type만 바꾸면 같은 walk-forward CV로 RF와 HGB를 바로 비교할 수 있다.
    calibrate=True면 TrendCalibrator로 파이프라인을 감싸 시즌 트렌드 외삽 보정을 적용한다.
    """
    if model_type == "rf":
        clf = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_leaf=200,
            n_jobs=-1,
            random_state=42,
        )
    elif model_type == "hgb":
        # [주의] `## 4-1.` 그리드서치로 찾은 lr=0.03/max_leaf_nodes=63/min_samples_leaf=400
        # 조합은 CV에서는 더 높았지만(981.79) 실제 제출은 767점으로 810점보다 하락해서
        # 되돌림 — CV 점수 히스토리 "그리드서치 결과가 실제 평가와 반대로 간 사례" 참고.
        clf = HistGradientBoostingClassifier(
            max_iter=100,           # RF의 n_estimators=100에 대응
            learning_rate=0.1,
            min_samples_leaf=200,   # RF와 동일한 잎 노드 최소 샘플로 정규화 강도를 맞춤
            early_stopping=False,   # walk-forward CV 자체가 검증 역할을 하므로 내부 조기종료는 끔
            random_state=42,
        )
    else:
        raise ValueError(f"알 수 없는 model_type: {model_type!r}")

    pipeline = Pipeline([
        ("asof_smooth", AsofRateSmoother()),
        ("derived", DerivedFeatureBuilder()),
        ("pre", preprocessor),
        ("clf", clf),
    ])
    return TrendCalibrator(pipeline) if calibrate else pipeline  # [신규]


def brier_skill_score(y_true, y_pred):
    """기존 5번 섹션에 있던 계산식을 함수로 분리 — fold마다 재사용."""
    r = y_true.mean()
    brier = ((y_pred - y_true) ** 2).mean()
    baseline_brier = r * (1 - r)
    return max(0, 100000 * (1 - brier / baseline_brier))


# [신규] 최근 시즌일수록 학습 가중치를 높인다 (half_life년마다 가중치 절반).
# 기준점은 학습 데이터의 마지막 시즌 = 예측 대상 바로 이전 해.
def season_sample_weight(seasons, half_life=2):
    max_season = seasons.max()
    return 0.5 ** ((max_season - seasons) / half_life)


# walk-forward fold 정의: (학습에 쓸 시즌들, 검증할 다음 시즌)
FOLDS = [
    (range(2019, 2022), 2022),  # train 2019~2021 -> val 2022
    (range(2019, 2023), 2023),  # train 2019~2022 -> val 2023
    (range(2019, 2024), 2024),  # train 2019~2023 -> val 2024 (기존 baseline과 동일)
]

# fold를 순회하며 매번 새 모델을 학습하고 점수를 기록
scores = []
for train_seasons, val_season in FOLDS:
    is_train = train["season"].isin(train_seasons)
    is_val = train["season"] == val_season
    X_train, y_train = train.loc[is_train, FEATURES], train.loc[is_train, TARGET]
    X_val, y_val = train.loc[is_val, FEATURES], train.loc[is_val, TARGET]

    fold_model = make_model()
    t = time.time()
    train_weight = season_sample_weight(train.loc[is_train, "season"])  # [신규]
    fold_model.fit(X_train, y_train, clf__sample_weight=train_weight)  # [변경]
    val_pred = fold_model.predict_proba(X_val)[:, 1]
    score = brier_skill_score(y_val, val_pred)
    scores.append(score)

    print(f"[val={val_season}] train={len(X_train)} val={len(X_val)} "
          f"score={score:.2f} ({time.time() - t:.1f}s)")

# fold 평균/표준편차로 점수 안정성 확인 (한 시즌만 봤을 때보다 신뢰도 높음)
mean_score = sum(scores) / len(scores)
std_score = (sum((s - mean_score) ** 2 for s in scores) / len(scores)) ** 0.5
print(f"\nCV Score: {mean_score:.2f} (mean) ± {std_score:.2f} (std) | "
      f"fold별: {[round(s, 2) for s in scores]} | model_type={MODEL_TYPE} | "
      f"calibrate={USE_TREND_CALIBRATION}")

# 다음 섹션(5. 전체 재학습)에서 쓸 파이프라인 — 아직 학습 전 상태로 정의만 해둠
# (fold_model들과는 별개 객체. 다음 셀에서 전체 데이터로 fit 됨)
model = make_model()

## 5. 전체 데이터로 재학습 & 모델 저장

검증으로 성능을 확인했으니 이제 전체 학습 데이터로 다시 학습합니다.

**[변경]** 여기서도 CV와 동일하게 `season_sample_weight()`로 최근 시즌(2024) 비중을
높여서 학습합니다. 실제 평가 대상인 2025년이 2024년 바로 다음 해라, 2024와 가장 가까운
가중치 분포로 학습하는 게 CV 때와 같은 논리입니다.

학습한 파이프라인을 `./model/rf.pkl` 로 저장합니다. 이 파일을 추론용 `script.py`,
`requirements.txt` 와 함께 `baseline_submit.zip` 으로 묶으면 제출 준비가 끝납니다.

In [ ]:
t = time.time()
final_weight = season_sample_weight(train["season"])  # [신규] CV와 동일한 recency weighting
model.fit(train[FEATURES], train[TARGET], clf__sample_weight=final_weight)  # [변경]
print(f"재학습 완료 :: {time.time() - t:.1f}s")

# [신규] HistGradientBoostingClassifier는 fit 상태에 numpy Generator(PCG64) 객체를
# _feature_subsample_rng로 들고 있는데(predict에는 안 쓰임, fit 전용), 이게 pickle에
# 그대로 들어가면 채점 서버 numpy 버전이 학습 환경과 다를 때
# "not a known BitGenerator module" 에러로 unpickle이 깨진다. predict에 영향 없으니
# 저장 전에 제거한다.
# [변경] TrendCalibrator로 감쌌으면 실제 파이프라인은 model.pipeline에 있음
inner_pipeline = model.pipeline if isinstance(model, TrendCalibrator) else model
clf = inner_pipeline.named_steps["clf"]
if hasattr(clf, "_feature_subsample_rng"):
    del clf._feature_subsample_rng

os.makedirs("./model", exist_ok=True)
joblib.dump(model, "./model/rf.pkl", compress=3)
print("저장 완료: ./model/rf.pkl")